```python
python -c "import torch; print(f'PyTorch CUDA Version: {torch.version.cuda}')"
python -c "import flash_attn; print(f'Flash Attention version: {flash_attn.__version__}')"
```

### glibc

- `ImportError: /lib/x86_64-linux-gnu/libc.so.6: version GLIBC_2.32 not found`
    - pip 默认安装的是预编译好的 .whl 文件。这个文件可能是在较新的 Linux 系统（如 Ubuntu 20.04 或更新版本）上构建的，依赖了较新版本的 glibc（C 运行库）。
    - 而当前自带的glibc 版本低于 2.32。
        - `ldd --version`
    - 当你使用 `--no-binary` 在本地编译时，编译器（gcc/nvcc）会针对你当前的操作系统环境（即 Ubuntu 20.04 和 glibc 2.31）链接动态库。
        - `pip install flash-attn==2.8.1 --no-binary flash-attn --no-build-isolation --no-cache-dir`

- https://github.com/Dao-AILab/flash-attention/issues/1708
    - 2.7.4.post1 的 flash_attn 依赖较低的 torch 版本，需要首先降级安装
        - `pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126`
    - `pip install flash_attn==2.7.4.post1`
    - `pip install flash_attn==2.7.4.post1 --no-build-isolation --no-cache-dir -v`

### flash-attn 源码编译

- `pip install packaging ninja`
    - check: ninja --version
    - `MAX_JOBS=4 pip install flash-attn --no-build-isolation`
        - `pip install flash-attn==2.8.1 --no-build-isolation --no-cache-dir -v`

```python
import torch
import flash_attn
print(f"Torch version: {torch.__version__}")
print(f"Flash attention version: {flash_attn.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device capability: {torch.cuda.get_device_capability()}")
    print(f"Device name: {torch.cuda.get_device_name()}")
try:
    from flash_attn import flash_attn_func
    q = torch.randn(1, 10, 32, 128, device='cuda', dtype=torch.float16)
    k = torch.randn(1, 10, 32, 128, device='cuda', dtype=torch.float16)
    v = torch.randn(1, 10, 32, 128, device='cuda', dtype=torch.float16)
    out = flash_attn_func(q, k, v)
    print("Flash attention test passed!")
except Exception as e:
    print(f"Flash attention test failed: {e}")
```

### vllm

- PyTorch 2.6 对应的官方 vLLM 预编译版本是 0.8.x 系列；
- 从 0.9.0 起官方 wheel 是按 torch 2.7 编译的，直接 pip install vllm==0.9.x 会把 torch 一起升到 2.7+。